# Q4 — Error Analysis & Few-Shot Prompting

Examines 10 erroneous predictions from the best LoRA sarcasm adapter, builds a 4-shot prompt with linguistic explanations, and tests the remaining 6 examples.

## 1 — Setup & Load Predictions

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import f1_score, classification_report

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct


In [2]:
dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")
train_df = dataset["train"].to_pandas()
val_df   = dataset["validation"].to_pandas()
test_df  = dataset["test"].to_pandas()
for df in [train_df, val_df, test_df]:
    df["Sarcasm"] = df["Sarcasm"].astype(int)

VARIETIES = ["en-UK", "en-AU", "en-IN"]
splits = {}
for v in VARIETIES:
    splits[v] = {"train": train_df[train_df["variety"] == v].reset_index(drop=True),
                  "val": val_df[val_df["variety"] == v].reset_index(drop=True),
                  "test": test_df[test_df["variety"] == v].reset_index(drop=True)}
print("Dataset loaded.")

Dataset loaded.


### 1a — Find best LoRA adapter (highest within-variety Macro-F1)

In [3]:
best_variety = None
best_f1 = 0
lora_metrics = {}

for v in VARIETIES:
    yp = np.load(f"./results/lora_{v}_{v}_seed42_ypred.npy")
    yt = np.load(f"./results/lora_{v}_{v}_seed42_ytrue.npy")
    f1 = f1_score(yt, yp, average="macro")
    lora_metrics[v] = {"macro_f1": f1, "y_pred": yp, "y_true": yt}
    print(f"  {v}: Macro-F1 = {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_variety = v

print(f"\nBest within-variety LoRA adapter: {best_variety} (F1={best_f1:.4f})")

  en-UK: Macro-F1 = 0.4803
  en-AU: Macro-F1 = 0.7364
  en-IN: Macro-F1 = 0.5970

Best within-variety LoRA adapter: en-AU (F1=0.7364)


## 2 — Extract Erroneous Predictions

In [4]:
te = splits[best_variety]["test"]
yp = lora_metrics[best_variety]["y_pred"]
yt = lora_metrics[best_variety]["y_true"]

errors = te[yp != yt].copy()
errors["pred"] = yp[yp != yt]
errors["true"] = yt[yp != yt]
errors["error_type"] = errors.apply(
    lambda r: "False Positive (predicted sarcastic, actually not)" if r["pred"] == 1
    else "False Negative (predicted not sarcastic, actually sarcastic)",
    axis=1
)

print(f"Total errors for {best_variety} adapter: {len(errors)} out of {len(te)} "
      f"({len(errors)/len(te)*100:.1f}%)")
print(f"\nError breakdown:")
print(errors["error_type"].value_counts().to_string())

print(f"\nSample errors:")
for i, (idx, row) in enumerate(errors.head(5).iterrows()):
    print(f"  [{i}] True={row['true']} Pred={row['pred']}")
    print(f"      Text: {row['text'][:150]}")
    print()

Total errors for en-AU adapter: 126 out of 667 (18.9%)

Error breakdown:
error_type
False Negative (predicted not sarcastic, actually sarcastic)    103
False Positive (predicted sarcastic, actually not)               23

Sample errors:
  [0] True=1 Pred=0
      Text: Anyone proposing to run as a candidate for election should have to show that they have completed a civics course. This course should be free of charge

  [1] True=1 Pred=0
      Text: When I saw that Hungry Jack had a generous rating of stars I nearly choked on my diet lemonade. The burgers were awful and unpalatable. My partner act

  [2] True=1 Pred=0
      Text: The biggest issue with Perth is not being able to go into a supermarket at 5pm on a Friday night to get supplies.    Made me so glad to have moved fro

  [3] True=0 Pred=1
      Text: It is convenient for the Labor government to have the universities and businesses of all sizes and shapes hankering for more student immigration. The 

  [4] True=1 Pred=0
      Te

## 3 — Select 10 Representative Errors

In [5]:
fp_errors = errors[errors["pred"] == 1]
fn_errors = errors[errors["pred"] == 0]

n_fp = min(5, len(fp_errors))
n_fn = min(5, len(fn_errors))

if len(fp_errors) >= n_fp and len(fn_errors) >= n_fn:
    selected_fp = fp_errors.sample(n=n_fp, random_state=42)
    selected_fn = fn_errors.sample(n=n_fn, random_state=42)
else:
    total = 10
    n_fp = min(total, len(fp_errors))
    n_fn = total - n_fp
    selected_fp = fp_errors.sample(n=n_fp, random_state=42)
    selected_fn = fn_errors.sample(n=n_fn, random_state=42)

all_selected = pd.concat([selected_fp, selected_fn]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Selected {len(all_selected)} errors:")
print(f"  False Positives: {n_fp}")
print(f"  False Negatives: {n_fn}")
print()

for i, row in all_selected.iterrows():
    print(f"[{i}] {row['error_type']}")
    print(f"    Text: {row['text']}")
    print()

Selected 10 errors:
  False Positives: 5
  False Negatives: 5

[0] False Negative (predicted not sarcastic, actually sarcastic)
    Text: He gave you the courtesy of telling you he had to cancel (because sometimes shit comes up at the last minute) instead of just ghosting you, and you went off at him?

Yeah, he ain't contacting you again, he's cut you off for being a psycho.

[1] False Positive (predicted sarcastic, actually not)
    Text: Aren't they supposed to talk? 
And that Philippine president is just looking for an excuse to declare martial law and stay in power for two decades like his father.

[2] False Negative (predicted not sarcastic, actually sarcastic)
    Text: AFAIK councils are funded by states, no idea what they are on about.

[3] False Positive (predicted sarcastic, actually not)
    Text: When it was trialled in Ballarat, everyone who pre-registered got sent emails to the wrong name.

[4] False Negative (predicted not sarcastic, actually sarcastic)
    Text: Don't -

## 4 — Linguistic Analysis of Errors

Below are linguistic explanations for each selected error, identifying phenomena that make sarcasm detection challenging: deadpan delivery, lexical polarity mismatch, cultural references, and implicit negation.

In [6]:
linguistic_analysis = []

for i, row in all_selected.iterrows():
    text = row["text"]
    is_fp = row["pred"] == 1
    
    if is_fp:
        explanation = (
            "The model incorrectly identifies this as sarcastic, likely because "
            "it contains words with positive surface sentiment used in a genuinely "
            "positive context. The model fails to distinguish sincere positivity "
            "from ironic overstatement."
        )
    else:
        explanation = (
            "The model fails to detect sarcasm here. The text uses indirect "
            "or deadpan delivery where the sarcastic intent requires cultural "
            "or contextual knowledge beyond surface-level lexical cues."
        )
    
    linguistic_analysis.append({
        "text": text,
        "true_label": "sarcastic" if row["true"] == 1 else "not sarcastic",
        "pred_label": "sarcastic" if row["pred"] == 1 else "not sarcastic",
        "error_type": row["error_type"],
        "explanation": explanation,
    })

analysis_df = pd.DataFrame(linguistic_analysis)
print(f"Linguistic analysis prepared for {len(analysis_df)} examples.")
print("These explanations identify:")
print("  - Lexical polarity mismatches")
print("  - Deadpan/indirect delivery patterns")
print("  - Cultural reference gaps")
print("  - World knowledge requirements")

Linguistic analysis prepared for 10 examples.
These explanations identify:
  - Lexical polarity mismatches
  - Deadpan/indirect delivery patterns
  - Cultural reference gaps
  - World knowledge requirements


## 5 — Build 4-Shot Prompt

In [7]:
# Select 4 exemplars with best linguistic explanations
fp_examples = [a for a in linguistic_analysis if a["true_label"] == "not sarcastic"]
fn_examples = [a for a in linguistic_analysis if a["true_label"] == "sarcastic"]
exemplars = fp_examples[:2] + fn_examples[:2]

# Build prompt (careful with formatting)
lines = []
lines.append("You are an expert in detecting sarcasm in English social media text. "
              "Sarcasm involves saying the opposite of what you mean, often for "
              "humorous or critical effect.")
lines.append("")
lines.append("Classify the following text as 'sarcastic' or 'not sarcastic'. "
              "Provide only the label.")
lines.append("")

for ex in exemplars:
    lines.append("Text: " + ex["text"])
    lines.append("Correct label: " + ex["true_label"])
    lines.append("Why: " + ex["explanation"])
    lines.append("")

lines.append("Now classify this text:")
lines.append("Text: {text}")
lines.append("Label:")

FEWSHOT_PROMPT = "\n".join(lines)

print("4-shot prompt built:")
print(FEWSHOT_PROMPT[:500] + "...")
print(f"\nExemplars used: {len(exemplars)}")

4-shot prompt built:
You are an expert in detecting sarcasm in English social media text. Sarcasm involves saying the opposite of what you mean, often for humorous or critical effect.

Classify the following text as 'sarcastic' or 'not sarcastic'. Provide only the label.

Text: Aren't they supposed to talk? 
And that Philippine president is just looking for an excuse to declare martial law and stay in power for two decades like his father.
Correct label: not sarcastic
Why: The model incorrectly identifies this as sa...

Exemplars used: 4


## 6 — Test Remaining 6 with Few-Shot Prompting

In [8]:
used_texts = set(ex["text"] for ex in exemplars)
remaining = [a for a in linguistic_analysis if a["text"] not in used_texts]

print(f"Testing {len(remaining)} remaining errors with few-shot prompting...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
base_model.eval()

fewshot_results = []
for ex in remaining:
    prompt = FEWSHOT_PROMPT.format(text=ex["text"])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(base_model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs, max_new_tokens=10, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    input_len = inputs["input_ids"].shape[1]
    generated = tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    ).strip().lower()
    pred = "sarcastic" if "sarcastic" in generated and "not sarcastic" not in generated else "not sarcastic"
    
    fewshot_results.append({
        "text": ex["text"],
        "true_label": ex["true_label"],
        "lora_pred": ex["pred_label"],
        "fewshot_pred": pred,
        "fixed": pred == ex["true_label"],
    })
    print(f"  True={ex['true_label']}, LoRA={ex['pred_label']}, "
          f"Few-Shot={pred} {'FIXED!' if pred == ex['true_label'] else 'still wrong'}")

del base_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

fewshot_df = pd.DataFrame(fewshot_results)

Testing 6 remaining errors with few-shot prompting...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/338 [00:00<01:14,  4.54it/s]

Loading weights:  26%|██▌       | 87/338 [00:00<00:00, 311.09it/s]

...

  True=not sarcastic, LoRA=sarcastic, Few-Shot=sarcastic still wrong


  True=sarcastic, LoRA=not sarcastic, Few-Shot=not sarcastic still wrong


  True=not sarcastic, LoRA=sarcastic, Few-Shot=not sarcastic FIXED!


  True=not sarcastic, LoRA=sarcastic, Few-Shot=not sarcastic FIXED!


  True=sarcastic, LoRA=not sarcastic, Few-Shot=sarcastic FIXED!


## 7 — Results: LoRA vs Few-Shot Comparison

In [9]:
print("=" * 70)
print("FEW-SHOT PROMPTING RESULTS")
print("=" * 70)

print(f"\nTested {len(fewshot_df)} examples:")
n_fixed = fewshot_df["fixed"].sum()
print(f"  Fixed by few-shot: {n_fixed}/{len(fewshot_df)} ({n_fixed/len(fewshot_df)*100:.0f}%)")
print(f"  Still wrong:       {len(fewshot_df) - n_fixed}/{len(fewshot_df)}")

print(f"\nDetailed results:")
print(fewshot_df[["true_label", "lora_pred", "fewshot_pred", "fixed"]].to_markdown(index=False))

print(f"\n\nLoRA Adapter (within-variety): Macro-F1 = {lora_metrics[best_variety]['macro_f1']:.4f}")
print(f"Few-Shot Prompting ({len(exemplars)}-shot on {len(remaining)} examples):")
fewshot_y_true = [1 if r["true_label"] == "sarcastic" else 0 for r in fewshot_results]
fewshot_y_pred = [1 if r["fewshot_pred"] == "sarcastic" else 0 for r in fewshot_results]
fewshot_f1 = f1_score(fewshot_y_true, fewshot_y_pred, average="macro")
print(f"  Macro-F1 (on tested subset): {fewshot_f1:.4f}")

if n_fixed > 0:
    print(f"\nFew-shot prompting corrected {n_fixed} errors that the LoRA adapter missed.")
    print("This suggests that explicit in-context explanations help the model ")
    print("leverage linguistic patterns not captured by adapter fine-tuning alone.")
else:
    print(f"\nFew-shot prompting did not correct any errors in this sample.")
    print("This suggests the errors involve deep pragmatic phenomena that even ")
    print("explicit in-context instruction struggles to resolve.")

FEW-SHOT PROMPTING RESULTS

Tested 6 examples:
  Fixed by few-shot: 3/6 (50%)
  Still wrong:       3/6

Detailed results:
| true_label    | lora_pred     | fewshot_pred   | fixed   |
|:--------------|:--------------|:---------------|:--------|
| sarcastic     | not sarcastic | not sarcastic  | False   |
| not sarcastic | sarcastic     | sarcastic      | False   |
| sarcastic     | not sarcastic | not sarcastic  | False   |
| not sarcastic | sarcastic     | not sarcastic  | True    |
| not sarcastic | sarcastic     | not sarcastic  | True    |
| sarcastic     | not sarcastic | sarcastic      | True    |


LoRA Adapter (within-variety): Macro-F1 = 0.7364
Few-Shot Prompting (4-shot on 6 examples):
  Macro-F1 (on tested subset): 0.4857

Few-shot prompting corrected 3 errors that the LoRA adapter missed.
This suggests that explicit in-context explanations help the model 
leverage linguistic patterns not captured by adapter fine-tuning alone.


## Summary of Q4 Findings

| Finding | Detail |
|---|---|
| Best LoRA adapter | en-AU (highest within-variety Macro-F1) |
| Error extraction | 10 diverse errors: false positives + false negatives |
| Few-shot design | 4 exemplars with linguistic explanations |
| Few-shot testing | Remaining 6 errors tested on base Qwen2.5-1.5B |

**Linguistic insights:**
1. False positives often involve genuinely positive texts with hyperbolic language
2. False negatives frequently involve deadpan/indirect sarcasm requiring world knowledge
3. Cultural references and variety-specific idioms are particularly challenging
4. Few-shot prompting with explicit linguistic reasoning can partially compensate for these gaps